In [34]:
from pathlib import Path
import re
import pandas as pd

# --- SETTINGS ---
gmt_files = [
    "/mnt/c/Users/jonan/Documents/1Work/RoseLab/References/mh.all.v2025.1.Mm.symbols.gmt",
    "/mnt/c/Users/jonan/Documents/1Work/RoseLab/References/m2.cp.v2025.1.Mm.symbols.gmt",
    "/mnt/c/Users/jonan/Documents/1Work/RoseLab/References/m5.go.bp.v2025.1.Mm.symbols.gmt",
    "/mnt/c/Users/jonan/Documents/1Work/RoseLab/References/m7.all.v2025.1.Mm.symbols.gmt",
]

out_csv = "/mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/references/cytotoxic_gene_sets_shortlist_noGenes_match.csv"

In [35]:
# --- LOAD GMT ---
def load_gmt(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            parts = ln.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            name, desc, *genes = parts
            rows.append({
                "set_name": name,
                "description": desc or "",
                "n_genes": len(genes),
                "genes": ";".join(genes),
                "source_file": Path(path).name,
            })
    return pd.DataFrame(rows)

df = pd.concat([load_gmt(p) for p in gmt_files], ignore_index=True)

# --- CYTOTOXICITY KEYWORDS (case-insensitive) ---
CYTO_KEYWORDS = [
    "cytotoxic", "cytotox", "cytolysis", "cell_killing", "cell killing",
    "natural killer", "nk cell", "t cell mediated cytotoxicity",
    "leukocyte mediated cytotoxicity", r"\bCTL\b", "granzyme", "perforin",
    "degranulation", "ADCC", "antibody-dependent", "FASL", "TRAIL"
]
pattern = re.compile("|".join(CYTO_KEYWORDS), flags=re.I)

# --- BOOLEAN MASK (no .apply; avoid match objects) ---
mask = (
    df["set_name"].str.contains(pattern, na=False) |
    df["description"].str.contains(pattern, na=False) #|
    # df["genes"].str.contains(pattern, na=False)
)

cyto_df = df.loc[mask].copy()

# Optional: sort & de-dup if the same set appears in multiple files
cyto_df.sort_values(["source_file", "set_name"], inplace=True)
cyto_df.drop_duplicates(subset=["set_name", "source_file"], inplace=True)

In [33]:
pattern

re.compile(r'cytotoxic|cytotox|cytolysis|cell_killing|cell killing|natural killer|nk cell|t cell mediated cytotoxicity|leukocyte mediated cytotoxicity|\bCTL\b|granzyme|perforin|degranulation|ADCC|antibody-dependent|FASL|TRAIL',
           re.IGNORECASE|re.UNICODE)

In [36]:
cyto_df

,set_name,description,n_genes,genes,source_file
279,BIOCARTA_TCYTOTOXIC_PATHWAY,https://www.gsea-msigdb.org/gsea/msigdb/mouse/...,12,Cd2;Cd247;Cd28;Cd3d;Cd3e;Cd3g;Cd8a;Icam1;Itgal...,m2.cp.v2025.1.Mm.symbols.gmt
1014,REACTOME_NEUTROPHIL_DEGRANULATION,https://www.gsea-msigdb.org/gsea/msigdb/mouse/...,534,1600012H06Rik;2310033P09Rik;A1bg;AY761185;Abca...,m2.cp.v2025.1.Mm.symbols.gmt
1526,REACTOME_TRAIL_SIGNALING,https://www.gsea-msigdb.org/gsea/msigdb/mouse/...,5,Casp8;Cflar;Fadd;Tnfrsf10b;Tnfsf10,m2.cp.v2025.1.Mm.symbols.gmt
2013,GOBP_ANTIBODY_DEPENDENT_CELLULAR_CYTOTOXICITY,https://www.gsea-msigdb.org/gsea/msigdb/mouse/...,7,Fcgr1;Fcgr2b;Fcgr3;Fcgr4;H2-T23;Ighe;Ighg1,m5.go.bp.v2025.1.Mm.symbols.gmt
2585,GOBP_CELL_KILLING,https://www.gsea-msigdb.org/gsea/msigdb/mouse/...,288,2410137M14Rik;Ager;Ap1g1;Apol11a;Arg1;Arl8b;Ar...,m5.go.bp.v2025.1.Mm.symbols.gmt
2775,GOBP_COMPLEMENT_DEPENDENT_CYTOTOXICITY,https://www.gsea-msigdb.org/gsea/msigdb/mouse/...,11,C3;Cd55;Cd59a;Cd59b;Cd5l;Cfh;Cr1l;Hsp90ab1;Il1...,m5.go.bp.v2025.1.Mm.symbols.gmt
2846,GOBP_CYTOLYSIS,https://www.gsea-msigdb.org/gsea/msigdb/mouse/...,10,Camp;Ccl28;Gbp2;Gbp2b;Gbp3;Gbp5;Gbp7;Igtp;Ninj...,m5.go.bp.v2025.1.Mm.symbols.gmt
2847,GOBP_CYTOLYSIS_BY_HOST_OF_SYMBIONT_CELLS,https://www.gsea-msigdb.org/gsea/msigdb/mouse/...,7,Apol11a;Camp;F2;Hrg;Ltf;Romo1;Spag11b,m5.go.bp.v2025.1.Mm.symbols.gmt
2848,GOBP_CYTOLYSIS_IN_ANOTHER_ORGANISM,https://www.gsea-msigdb.org/gsea/msigdb/mouse/...,7,Ccl28;Gbp2;Gbp2b;Gbp3;Gbp5;Gbp7;Igtp,m5.go.bp.v2025.1.Mm.symbols.gmt
2863,GOBP_CYTOTOXIC_T_CELL_DIFFERENTIATION,https://www.gsea-msigdb.org/gsea/msigdb/mouse/...,6,Cd8a;Hsp90aa1;Lilrb4a;Lilrb4b;Sart1;Tnfsf9,m5.go.bp.v2025.1.Mm.symbols.gmt


In [37]:
# --- SAVE CSV ---
cyto_df.to_csv(out_csv, index=False)
print(f"Saved {len(cyto_df)} cytotoxic-related pathways to {out_csv}")

Saved 52 cytotoxic-related pathways to /mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/references/cytotoxic_gene_sets_shortlist_noGenes_match.csv


In [32]:
# --- LOADERS ---
def load_gmt(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            parts = ln.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            name, desc, *genes = parts
            rows.append({
                "set_name": name,
                "description": desc or "",
                "genes_list": genes,
                "n_genes": len(genes),
                "genes": ";".join(genes),
                "source_file": Path(path).name
            })
    return pd.DataFrame(rows)

df = pd.concat([load_gmt(p) for p in gmt_files], ignore_index=True)

# --- COLLECTION TAGS (helps provenance sorting) ---
def tag_collection(name):
    if name.startswith("HALLMARK_"): return "Hallmark"
    if name.startswith("KEGG_"):     return "KEGG"
    if name.startswith("REACTOME_"): return "Reactome"
    if name.startswith("BIOCARTA_"): return "BioCarta"
    if name.startswith("GOBP_"):     return "GO:BP"
    if name.startswith("GOCC_"):     return "GO:CC"
    if name.startswith("GOMF_"):     return "GO:MF"
    if name.startswith("IMMUNESIGDB_") or name.startswith("IMM_") or name.startswith("C7_") or name.startswith("M7_"):
        return "ImmuneSigDB"
    return "Other"
df["collection"] = df["set_name"].map(tag_collection)

# --- CORE CYTOTOX KEYS (names/descriptions must match one of these) ---
NAME_KEYS = [
    r"\bcytotoxic\b", r"\bcytolysis\b", r"\bcell killing\b", r"\blytic granule\b",
    r"\bdegranulation\b",  # allowed but will be filtered by allow/block rules
    r"\bnatural killer\b", r"\bnk cell\b", r"\bt cell mediated cytotoxicity\b",
    r"\bleukocyte mediated cytotoxicity\b", r"\bantibody[-\s]?dependent cellular cytotoxicity\b",
    r"\bADCC\b", r"\bCTL\b"
]
name_pat = re.compile("|".join(NAME_KEYS), re.I)

# Secondary cues allowed in names/descriptions (death receptor routes)
SECONDARY_NAME_KEYS = [r"\bperforin\b", r"\bgranzyme\b", r"\bFASL?\b", r"\bTRAIL\b"]
secondary_pat = re.compile("|".join(SECONDARY_NAME_KEYS), re.I)

# --- MARKER GENE LIST (for secondary rule) ---
KILLER_MARKERS = {"PRF1","GZMB","GZMH","GZMA","GNLY","NKG7","CTSW","KLRK1","KLRD1","GZMK","GZMM"}
def killer_marker_hits(genes):
    return len(set(genes) & KILLER_MARKERS)

# --- BLOCKLIST: exclude generic “positive regulation of X” and unrelated processes ---
BLOCK_NAME_SUBSTRINGS = [
    "POSITIVE_REGULATION_OF_SIGNAL_TRANSDUCTION",
    "POSITIVE_REGULATION_OF_INTRACELLULAR_SIGNAL_TRANSDUCTION",
    "POSITIVE_REGULATION_OF_CELL_POPULATION_PROLIFERATION",
    "POSITIVE_REGULATION_OF_LIPID_TRANSPORT",
    "POSITIVE_REGULATION_OF_LIPID_LOCALIZATION",
    "POSITIVE_REGULATION_OF_PHOSPHOLIPID_TRANSPORT",
    "ERBB_SIGNALING_PATHWAY",
    "NEURON_APOPTOTIC_PROCESS",
    "ENDOTHELIAL_CELL_APOPTOTIC_PROCESS",
]
def blocked(name):
    up = name.upper()
    return any(b in up for b in BLOCK_NAME_SUBSTRINGS)

# --- ALLOWLIST for degranulation contexts (keep NK/CTL/neutrophil only when explicitly immune/cytotoxic) ---
ALLOW_IF_DEGRANULATION_IN_NAME = [
    "NK", "NATURAL_KILLER", "CTL", "CYTOTOX", "CYTOLYSIS", "LEUKOCYTE_MEDIATED_CYTOTOXICITY",
    "T_CELL_MEDIATED_CYTOTOXICITY", "ANTIBODY_DEPENDENT_CELLULAR_CYTOTOXICITY"
]

def select_row(row):
    name = row["set_name"]
    desc = row["description"]
    genes = row["genes_list"]
    coll = row["collection"]

    # rule 0: fast block
    if blocked(name):
        return False, "blocked_generic_regulation"

    # rule 1: explicit name/description match
    if name_pat.search(name) or name_pat.search(desc):
        # If term is about "degranulation", require immune/cytotoxic context in the name
        if re.search(r"\bdegranulation\b", name, re.I) and not any(s in name.upper() for s in ALLOW_IF_DEGRANULATION_IN_NAME):
            return False, "degranulation_without_cytotoxic_context"
        return True, "explicit_name_desc"

    # rule 2: secondary keywords in name/desc (perforin/granzyme/FASL/TRAIL) but avoid very broad terms
    if secondary_pat.search(name) or secondary_pat.search(desc):
        return True, "secondary_route_in_name_desc"

    # rule 3: gene-membership support (>=2 killer markers) AND collection is curated immune/pathway (GO:BP/KEGG/Reactome/BioCarta)
    if killer_marker_hits(genes) >= 2 and coll in {"GO:BP","KEGG","Reactome","BioCarta"}:
        # still avoid super generic GO terms
        if "POSITIVE_REGULATION" in name.upper() or "SIGNAL_TRANSDUCTION" in name.upper():
            return False, "generic_go_with_markers"
        return True, f"gene_membership_{killer_marker_hits(genes)}markers"

    return False, "no_match"

sel, reason = [], []
for _, r in df.iterrows():
    keep, why = select_row(r)
    sel.append(keep)
    reason.append(why)

cur = df.loc[sel].copy()
cur["match_reason"] = reason

# tidy up & export
cur = cur[["set_name","collection","n_genes","source_file","match_reason","description","genes"]]
cur.sort_values(["collection","set_name"], inplace=True)
cur.to_csv(out_csv, index=False)
print(f"Saved {len(cur)} curated cytotoxic-related pathways to {out_csv}")

Saved 10445 curated cytotoxic-related pathways to /mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/references/cytotoxic_gene_sets_shortlist.csv
